In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print('Libraries imported successfully!')

Libraries imported successfully!


In [2]:
df = pd.read_csv('../data/retail_store_inventory.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

Dataset loaded successfully!
Shape: (73100, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## 2. Convert Date and sort the time series

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
df[['Date', 'Store ID', 'Product ID', 'Units Sold']].head(10)

,Date,Store ID,Product ID,Units Sold
0,2022-01-01,S001,P0001,127
1,2022-01-02,S001,P0001,81
2,2022-01-03,S001,P0001,5
3,2022-01-04,S001,P0001,58
4,2022-01-05,S001,P0001,147
5,2022-01-06,S001,P0001,37
6,2022-01-07,S001,P0001,107
7,2022-01-08,S001,P0001,2
8,2022-01-09,S001,P0001,350
9,2022-01-10,S001,P0001,36


## 3. Create calendar features

In [4]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df[['Date','Year','Month','Day','Day_of_Week','Week_of_Year']].head(10)

,Date,Year,Month,Day,Day_of_Week,Week_of_Year
0,2022-01-01,2022,1,1,5,52
1,2022-01-02,2022,1,2,6,52
2,2022-01-03,2022,1,3,0,1
3,2022-01-04,2022,1,4,1,1
4,2022-01-05,2022,1,5,2,1
5,2022-01-06,2022,1,6,3,1
6,2022-01-07,2022,1,7,4,1
7,2022-01-08,2022,1,8,5,1
8,2022-01-09,2022,1,9,6,1
9,2022-01-10,2022,1,10,0,2


## 4. Define Store + Product groups

Historical demand must be calculated separately for every Store ID + Product ID combination.

In [5]:
group_cols = ['Store ID', 'Product ID']
print('Store-Product combinations:', df[group_cols].drop_duplicates().shape[0])

Store-Product combinations: 100


## 5. Create lag features

`Lag_1` = previous day, `Lag_7` = 7 days ago, `Lag_14` = 14 days ago, `Lag_28` = 28 days ago.

In [6]:
df['Lag_1'] = df.groupby(group_cols)['Units Sold'].shift(1)
df['Lag_7'] = df.groupby(group_cols)['Units Sold'].shift(7)
df['Lag_14'] = df.groupby(group_cols)['Units Sold'].shift(14)
df['Lag_28'] = df.groupby(group_cols)['Units Sold'].shift(28)
df[['Date','Store ID','Product ID','Units Sold','Lag_1','Lag_7','Lag_14','Lag_28']].head(35)

,Date,Store ID,Product ID,Units Sold,Lag_1,Lag_7,Lag_14,Lag_28
0,2022-01-01,S001,P0001,127,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,81,127.0,NaN,NaN,NaN
2,2022-01-03,S001,P0001,5,81.0,NaN,NaN,NaN
3,2022-01-04,S001,P0001,58,5.0,NaN,NaN,NaN
4,2022-01-05,S001,P0001,147,58.0,NaN,NaN,NaN
5,2022-01-06,S001,P0001,37,147.0,NaN,NaN,NaN
6,2022-01-07,S001,P0001,107,37.0,NaN,NaN,NaN
7,2022-01-08,S001,P0001,2,107.0,127.0,NaN,NaN
8,2022-01-09,S001,P0001,350,2.0,81.0,NaN,NaN
9,2022-01-10,S001,P0001,36,350.0,5.0,NaN,NaN


## 6. Create rolling demand features

`shift(1)` prevents the current target from entering its own rolling statistics.

In [7]:
df['Rolling_Mean_7'] = df.groupby(group_cols)['Units Sold'].transform(lambda x: x.shift(1).rolling(7).mean())
df['Rolling_Mean_14'] = df.groupby(group_cols)['Units Sold'].transform(lambda x: x.shift(1).rolling(14).mean())
df['Rolling_Mean_28'] = df.groupby(group_cols)['Units Sold'].transform(lambda x: x.shift(1).rolling(28).mean())
df[['Rolling_Mean_7','Rolling_Mean_14','Rolling_Mean_28']].head(35)

,Rolling_Mean_7,Rolling_Mean_14,Rolling_Mean_28
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,80.285714,NaN,NaN
8,62.428571,NaN,NaN
9,100.857143,NaN,NaN


## 7. Rolling demand volatility

In [8]:
df['Rolling_Std_7'] = df.groupby(group_cols)['Units Sold'].transform(lambda x: x.shift(1).rolling(7).std())
df[['Date','Store ID','Product ID','Units Sold','Rolling_Std_7']].head(35)

,Date,Store ID,Product ID,Units Sold,Rolling_Std_7
0,2022-01-01,S001,P0001,127,NaN
1,2022-01-02,S001,P0001,81,NaN
2,2022-01-03,S001,P0001,5,NaN
3,2022-01-04,S001,P0001,58,NaN
4,2022-01-05,S001,P0001,147,NaN
5,2022-01-06,S001,P0001,37,NaN
6,2022-01-07,S001,P0001,107,NaN
7,2022-01-08,S001,P0001,2,50.638307
8,2022-01-09,S001,P0001,350,53.384945
9,2022-01-10,S001,P0001,36,121.870736


## 8. Competitor price feature

In [9]:
df['Price_Difference'] = df['Price'] - df['Competitor Pricing']
df[['Price','Competitor Pricing','Price_Difference']].head(10)

,Price,Competitor Pricing,Price_Difference
0,33.50,29.69,3.81
1,27.95,30.89,-2.94
2,62.70,58.22,4.48
3,77.88,75.99,1.89
4,28.46,29.40,-0.94
5,91.05,87.30,3.75
6,53.99,58.80,-4.81
7,66.30,63.82,2.48
8,41.72,38.68,3.04
9,32.68,34.71,-2.03


## 9. Review engineered features

In [10]:
feature_columns = ['Date','Store ID','Product ID','Units Sold','Year','Month','Day','Day_of_Week','Week_of_Year','Lag_1','Lag_7','Lag_14','Lag_28','Rolling_Mean_7','Rolling_Mean_14','Rolling_Mean_28','Rolling_Std_7','Price_Difference']
df[feature_columns].head(35)

,Date,Store ID,Product ID,Units Sold,Year,Month,Day,Day_of_Week,Week_of_Year,Lag_1,Lag_7,Lag_14,Lag_28,Rolling_Mean_7,Rolling_Mean_14,Rolling_Mean_28,Rolling_Std_7,Price_Difference
0,2022-01-01,S001,P0001,127,2022,1,1,5,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.81
1,2022-01-02,S001,P0001,81,2022,1,2,6,52,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.94
2,2022-01-03,S001,P0001,5,2022,1,3,0,1,81.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.48
3,2022-01-04,S001,P0001,58,2022,1,4,1,1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.89
4,2022-01-05,S001,P0001,147,2022,1,5,2,1,58.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.94
5,2022-01-06,S001,P0001,37,2022,1,6,3,1,147.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.75
6,2022-01-07,S001,P0001,107,2022,1,7,4,1,37.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.81
7,2022-01-08,S001,P0001,2,2022,1,8,5,1,107.0,127.0,NaN,NaN,80.285714,NaN,NaN,50.638307,2.48
8,2022-01-09,S001,P0001,350,2022,1,9,6,1,2.0,81.0,NaN,NaN,62.428571,NaN,NaN,53.384945,3.04
9,2022-01-10,S001,P0001,36,2022,1,10,0,2,350.0,5.0,NaN,NaN,100.857143,NaN,NaN,121.870736,-2.03


## 10. Check NaNs created by lag/rolling features

NaNs at the beginning of each Store + Product series are expected.

In [11]:
time_series_features = ['Lag_1','Lag_7','Lag_14','Lag_28','Rolling_Mean_7','Rolling_Mean_14','Rolling_Mean_28','Rolling_Std_7']
print(df[time_series_features].isnull().sum())

Lag_1               100
Lag_7               700
Lag_14             1400
Lag_28             2800
Rolling_Mean_7      700
Rolling_Mean_14    1400
Rolling_Mean_28    2800
Rolling_Std_7       700
dtype: int64


## 11. Verify one Store + Product series

In [12]:
sample_series = df[(df['Store ID']=='S001') & (df['Product ID']=='P0001')][['Date','Units Sold','Lag_1','Lag_7','Lag_14','Lag_28','Rolling_Mean_7','Rolling_Mean_14','Rolling_Mean_28']].head(35)
sample_series

,Date,Units Sold,Lag_1,Lag_7,Lag_14,Lag_28,Rolling_Mean_7,Rolling_Mean_14,Rolling_Mean_28
0,2022-01-01,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-02,81,127.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-03,5,81.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-04,58,5.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-01-05,147,58.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2022-01-06,37,147.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2022-01-07,107,37.0,NaN,NaN,NaN,NaN,NaN,NaN
7,2022-01-08,2,107.0,127.0,NaN,NaN,80.285714,NaN,NaN
8,2022-01-09,350,2.0,81.0,NaN,NaN,62.428571,NaN,NaN
9,2022-01-10,36,350.0,5.0,NaN,NaN,100.857143,NaN,NaN


## Next steps

After verifying these features: remove rows with unavailable lag/rolling values, define `X` and `y`, encode categorical variables, create a chronological train/validation/test split, then train baseline and XGBoost models. Do not use random `train_test_split` for this forecasting task.